# Experiment 2: Provider-Agnostic LLM Client (CO1)
## One interface over a hosted API and a local Ollama model

**Aim:** Build one Python LLM client interface that can send chat requests to either a hosted OpenAI-compatible API or a locally running Ollama model.

**Learning outcomes**
- Explain hosted versus local LLM inference.
- Identify API endpoints, model identifiers, credentials, and chat messages.
- Implement a reusable `LLMClient` abstraction.
- Configure and test a hosted API and a local Ollama model.
- Switch providers without rewriting application logic.
- Compare latency, privacy, network dependency, and resource requirements.

**Architecture:** Application → `LLMClient.chat()` → configured provider endpoint → hosted API OR local Ollama → response.

**Important:** The hosted example uses a configurable Groq OpenAI-compatible endpoint. A hosted API key may be subject to quotas or charges; check current provider documentation. Never hard-code or commit keys. Ollama in this notebook runs inside the Colab virtual machine, not on your personal laptop. Colab GPU availability is not guaranteed.

## 0. Prepare Colab
1. Open Google Colab and create a notebook.
2. Select Runtime → Change runtime type. Choose GPU if available (CPU works for a small model but may be slow).
3. Run cells in order.
4. For the hosted section, create an API key with your chosen provider and store it in Colab Secrets as `GROQ_API_KEY`.
5. The local section installs Ollama in the Colab runtime and downloads a small model.

## 1. Install the OpenAI-compatible client

In [ ]:
!pip -q install -U openai

In [ ]:
import os, time, requests, subprocess
from openai import OpenAI
print("OpenAI-compatible SDK ready")

## 2. Implement the common client
The `LLMClient` class accepts an endpoint, API key, model identifier, and provider label. Its `chat()` method is the same for both providers.

In [ ]:
class LLMClient:
    def __init__(self, base_url, api_key, model, provider_name):
        self.client = OpenAI(base_url=base_url, api_key=api_key)
        self.model = model
        self.provider_name = provider_name

    def chat(self, user_message, system_message="You are a helpful assistant.",
             temperature=0.7, max_tokens=300):
        result = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return result.choices[0].message.content

print("LLMClient defined")

## 3. Configure the hosted API
Example endpoint: Groq's OpenAI-compatible API. Add a secret named `GROQ_API_KEY` in Colab's Secrets panel (key icon), and enable notebook access. If using another provider, change the base URL and model name according to that provider's current documentation.

In [ ]:
from google.colab import userdata

try:
    HOSTED_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    HOSTED_API_KEY = None

HOSTED_BASE_URL = "https://api.groq.com/openai/v1"
HOSTED_MODEL = "llama-3.3-70b-versatile"  # Check current provider model availability.

hosted_llm = None
if HOSTED_API_KEY:
    hosted_llm = LLMClient(
        HOSTED_BASE_URL, HOSTED_API_KEY, HOSTED_MODEL, "Hosted API"
    )
    print("Hosted client configured")
else:
    print("Add GROQ_API_KEY in Colab Secrets to enable hosted API test")

## 4. Test the hosted model

In [ ]:
if hosted_llm:
    print(hosted_llm.chat(
        "Explain AI and machine learning using a simple example.",
        temperature=0.3, max_tokens=220
    ))
else:
    print("Hosted test skipped: no API key configured.")

## 5. Install and start Ollama in Colab
Ollama provides a local model runtime. This cell installs it and starts the server on port 11434.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

ready = False
for _ in range(30):
    try:
        r = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

print("Ollama ready:", ready)
if not ready:
    raise RuntimeError("Ollama server did not start. Rerun this cell.")

## 6. Download a small local model
The `qwen2.5:0.5b` model is selected for a compact demonstration. Download time depends on network speed. The model must be downloaded again if the temporary Colab runtime is reset.

In [ ]:
!ollama pull qwen2.5:0.5b

## 7. Configure the local Ollama client
Notice: the `LLMClient` class is unchanged. Only its configuration differs.

In [ ]:
local_llm = LLMClient(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",  # Placeholder for the local Ollama endpoint
    model="qwen2.5:0.5b",
    provider_name="Local Ollama"
)
print("Local Ollama client configured")

## 8. Test the local model

In [ ]:
print(local_llm.chat(
    "Explain neural networks to a beginner with one practical example.",
    temperature=0.3, max_tokens=220
))

## 9. Demonstrate provider switching
The application function is provider-independent. Pass either configured client to `ask_llm()`.

In [ ]:
def ask_llm(client, question):
    return client.chat(
        question,
        system_message="You are a concise AI tutor. Explain clearly and use an example.",
        temperature=0.4,
        max_tokens=220
    )

question = "Differentiate supervised and unsupervised learning."
print("=== LOCAL OLLAMA ===")
print(ask_llm(local_llm, question))

if hosted_llm:
    print("\\n=== HOSTED API ===")
    print(ask_llm(hosted_llm, question))
else:
    print("\\nHosted test skipped: configure GROQ_API_KEY.")

## 10. Optional: Gradio provider selector
This interface lets the user choose Hosted API or Local Ollama. If the hosted key is absent, the application displays a setup message.

In [ ]:
!pip -q install -U gradio

In [ ]:
import gradio as gr

def gradio_chat(message, provider):
    if provider == "Hosted API":
        if hosted_llm is None:
            return "Configure GROQ_API_KEY in Colab Secrets, then rerun the hosted configuration cell."
        client = hosted_llm
    else:
        client = local_llm
    try:
        return ask_llm(client, message)
    except Exception as exc:
        return f"Request failed: {type(exc).__name__}: {exc}"

app = gr.Interface(
    fn=gradio_chat,
    inputs=[
        gr.Textbox(lines=3, label="Your question"),
        gr.Radio(["Local Ollama", "Hosted API"], value="Local Ollama",
                 label="Select provider")
    ],
    outputs=gr.Textbox(lines=8, label="Response"),
    title="Provider-Agnostic LLM Client",
    description="One interface for hosted inference and local Ollama."
)
app.launch(share=True, debug=True)

## 11. Compare the two providers
Complete the table with observations from your own run. Do not invent performance values.

| Criterion | Hosted API | Local Ollama |
|---|---|---|
| Execution | Provider infrastructure | Colab VM in this experiment |
| Internet for inference | Required | Not required after model download |
| API key | Usually required | No hosted API key |
| Prompt transmission | Sent to provider | Stays in Colab VM |
| Compute | Provider-managed | Colab runtime |
| Cost/limits | Provider quota/pricing | Runtime resources and usage limits |

### Optional timing test
A single measurement is not a reliable benchmark. Repeat several times and report the conditions.

In [ ]:
import time

def timed_query(client, question):
    start = time.perf_counter()
    answer = client.chat(question, max_tokens=120, temperature=0.3)
    return answer, time.perf_counter() - start

q = "Explain tokenization in natural language processing."
local_answer, local_time = timed_query(local_llm, q)
print(f"Local Ollama: {local_time:.2f} seconds")

if hosted_llm:
    hosted_answer, hosted_time = timed_query(hosted_llm, q)
    print(f"Hosted API: {hosted_time:.2f} seconds")
else:
    print("Hosted timing skipped: API key not configured.")

## 12. Student exercises

**Beginner**
1. Define provider-agnostic.
2. Identify the base URL, API key, and model name in hosted configuration.
3. Explain why Ollama uses a local endpoint.
4. Run the same prompt on the local model.

**Intermediate**
1. Change the system prompt to a Python programming tutor.
2. Compare outputs at temperature 0.2 and 1.0.
3. Add conversation history to the client.
4. Explain why API timeouts are useful.

**Advanced**
1. Add a third OpenAI-compatible provider by changing configuration only.
2. Implement retry handling for transient API errors.
3. Log provider, model, elapsed time, and success/failure without logging secrets or private prompts.
4. Design an optional hosted-to-local fallback and discuss privacy and response-quality implications.

## 13. Troubleshooting and safety

| Problem | Action |
|---|---|
| Hosted authentication error | Verify Colab Secret name, key, and notebook access |
| Hosted model not found | Check the provider's current model catalogue |
| Quota/rate limit | Check provider limits, billing, and retry policy |
| `ollama` command not found | Rerun installation cell |
| Ollama server not ready | Rerun server cell and check port 11434 |
| Model download fails | Check connectivity and retry `ollama pull` |
| Local inference slow | Use a smaller model or available GPU |
| Gradio link blocked | Network firewall may block public tunnels; request IT review |

Never commit API keys. Do not send confidential data to hosted services without authorization. Ollama here runs inside Colab, not on your laptop.

## 14. Result and viva
**Result:** A common `LLMClient` interface was configured for a hosted OpenAI-compatible API and a local Ollama model. Provider switching was demonstrated without rewriting the application-level call.

**Viva questions**
1. What does provider-agnostic mean?
2. What is an API endpoint?
3. Why is an API key used?
4. What is an OpenAI-compatible API?
5. What is Ollama?
6. Compare hosted and local inference.
7. Why is a client abstraction useful?
8. Why must secrets not be committed to GitHub?
9. What are the trade-offs of local inference?
10. How can an application support multiple providers?

**Suggested assessment (20 marks):** Architecture 3; common client implementation 5; hosted API setup 3; Ollama setup 4; provider switching 3; observations and viva 2.